# 01 — Exploratory Data Analysis

Sanity-check the corpus and clustering before trusting downstream insights.
Run **after** `python -m src.pipeline.run` (or `--no-llm`).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

df = pd.read_parquet(PROJECT_ROOT / 'data/processed/feedback.parquet')
print(df.shape)
df.head(3)

## Source balance, ratings, sentiment

In [ ]:
display(df['source'].value_counts())
ax = df['rating'].value_counts().sort_index().plot.bar(title='Star ratings (reviews only)')


In [ ]:
df.groupby('source')['vader_compound'].describe()

## Text length and timestamps

In [ ]:
df['text_clean'].str.len().plot.hist(bins=50, title='Cleaned text length (chars)')


In [ ]:
ts = pd.to_datetime(df['timestamp'], utc=True)
ts.groupby([df['source'], ts.dt.to_period('M')]).size().unstack(0).plot(
    title='Items per month per source', figsize=(10, 4))

## Cluster sizes and spot-check
Read a few items from the biggest clusters — do they hang together?

In [ ]:
sizes = df[df['cluster_id'] != -1]['cluster_id'].value_counts()
print(f"clusters: {len(sizes)}, noise: {(df['cluster_id'] == -1).mean():.0%}")
sizes.head(15).plot.bar(title='Largest clusters')

In [ ]:
cid = sizes.index[0]  # change to inspect other clusters
for t in df[df['cluster_id'] == cid]['text'].head(10):
    print('-', t[:160])

## Near-duplicate share per cluster
High near-dup share = one viral phrasing dominating; quoting excludes these.

In [ ]:
dup_share = df[df['cluster_id'] != -1].groupby('cluster_id')['is_near_dup'].mean()
dup_share.sort_values(ascending=False).head(10)